In [4]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [5]:
import torch
from src.config import DATASET_ROOT, POSE_DATASET_ROOT, GAMMA_POSE_DATASET_ROOT
from src.Skeleton_model.yolo_pose_tracking import save_annotated_pose_videos
from src.rwf2000 import RWF2000PoseDataset 
from src.Skeleton_model.graph import SkeletonGraph, compute_joint_distance_to_center_of_gravity
from src.Skeleton_model.stgcn import STGCN
from scripts.common.get_device import get_available_device
from ultralytics import YOLO
from pathlib import Path
import torch
import numpy as np
from src.Skeleton_model.yolo_pose_tracking import pose_data_to_stgcn_tensor

In [3]:
pose_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT, split="train")
radii = compute_joint_distance_to_center_of_gravity(pose_dataset)
skeleton_graph = SkeletonGraph(radii)
device = get_available_device()
model = STGCN(adjacency=skeleton_graph.A).to(device)

/mnt/vurm/homes/homes/mp2940/violence-detection-dissertation/src/Skeleton_model/graph.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.radii = torch.tensor(radii, dtype=torch.float32)


Using cuda:0 with 20.55 GB free


In [26]:
pose_path, label = pose_dataset.samples[10]
pose_data = torch.load(pose_path, weights_only=False)
print(pose_data["frames"][140]['people'][0]['keypoint_confidence'].sum())

tensor(13.9065)


In [8]:
from pathlib import Path
import torch
import numpy as np

def analyse_pose_database(pose_root):
    pose_files = list(Path(pose_root).rglob("*.pt"))

    empty_samples = 0
    detection_coverages = []       # from saved summary — true "YOLO saw someone" rate
    tracking_coverages = []        # from saved summary — true "tracker assigned an ID" rate
    frames_with_detections_list = []
    frames_without_track_ids_list = []
    visible_joints_per_frame = []
    keypoint_confidences = []
    unique_track_ids_per_video = []

    for pose_path in pose_files:
        pose_data = torch.load(pose_path, weights_only=False)
        frames = pose_data["frames"]
        summary = pose_data["summary"]

        total_visible_joints = 0
        track_ids = set()
        sample_has_pose = False

        for frame in frames:
            people = frame["people"]

            if len(people) > 0:
                sample_has_pose = True

            for person in people:
                track_ids.add(person["track_id"])

                confidence = person["keypoint_confidence"]
                visible = confidence > 0

                total_visible_joints += visible.sum().item()
                keypoint_confidences.extend(
                    confidence[visible].tolist()
                )

        num_frames = len(frames)

        if not sample_has_pose:
            empty_samples += 1

        # true detection coverage: frames where YOLO found >=1 person,
        # regardless of whether a track ID was later assigned
        detection_coverages.append(
            summary["detection_frame_coverage_percentage"]
        )

        # true tracking coverage: frames where a track ID was successfully assigned
        tracking_coverages.append(
            summary["tracking_frame_coverage_percentage"]
        )

        # raw counts, useful for spotting videos where detection succeeded
        # but tracking dropped the ball
        frames_with_detections_list.append(
            summary["frames_with_detections"]
        )
        frames_without_track_ids_list.append(
            summary["frames_without_track_ids"]
        )

        visible_joints_per_frame.append(
            total_visible_joints / num_frames
        )

        unique_track_ids_per_video.append(
            len(track_ids)
        )

    return {
        "num_samples": len(pose_files),
        "empty_samples": empty_samples,
        "mean_detection_coverage": np.mean(detection_coverages),
        "mean_tracking_coverage": np.mean(tracking_coverages),
        "mean_tracking_detection_gap": np.mean(
            np.array(detection_coverages) - np.array(tracking_coverages)
        ),
        "mean_frames_without_track_ids": np.mean(frames_without_track_ids_list),
        "mean_visible_joints_per_frame": np.mean(visible_joints_per_frame),
        "mean_keypoint_confidence": np.mean(keypoint_confidences),
        "mean_unique_track_ids_per_video": np.mean(unique_track_ids_per_video),
    }

In [9]:
from src.config import DATASET_ROOT, POSE_DATASET_ROOT, GAMMA_POSE_DATASET_ROOT, POSE_DATASET_ROOT_OCSORT, POSE_DATASET_ROOT_BYTETRACK, POSE_DATASET_ROOT_OCSORT2 
original_results = analyse_pose_database(
    POSE_DATASET_ROOT
)

ocsort_results = analyse_pose_database(
    POSE_DATASET_ROOT_OCSORT
)

print("BYTETRACK")
for key, value in original_results.items():
    print(f"{key}: {value}")

print("OCSORT")
for key, value in ocsort_results.items():
    print(f"{key}: {value}")


BYTETRACK
num_samples: 2000
empty_samples: 288
mean_detection_coverage: 0.7097066666666667
mean_tracking_coverage: 0.6915333333333334
mean_tracking_detection_gap: 0.01817333333333334
mean_frames_without_track_ids: 2.726
mean_visible_joints_per_frame: 28.619839999999996
mean_keypoint_confidence: 0.6205726204782711
mean_unique_track_ids_per_video: 6.101
OCSORT
num_samples: 2000
empty_samples: 288
mean_detection_coverage: 0.7097066666666667
mean_tracking_coverage: 0.6933833333333334
mean_tracking_detection_gap: 0.01632333333333334
mean_frames_without_track_ids: 2.4485
mean_visible_joints_per_frame: 28.821176666666666
mean_keypoint_confidence: 0.6197176046431078
mean_unique_track_ids_per_video: 5.7035


In [10]:
original_results = analyse_pose_database(
    POSE_DATASET_ROOT_BYTETRACK 
)

ocsort_results = analyse_pose_database(
    POSE_DATASET_ROOT_OCSORT2
)

print("BYTETRACK")
for key, value in original_results.items():
    print(f"{key}: {value}")

print("OCSORT")
for key, value in ocsort_results.items():
    print(f"{key}: {value}")

BYTETRACK
num_samples: 2000
empty_samples: 286
mean_detection_coverage: 0.7097066666666667
mean_tracking_coverage: 0.6938500000000001
mean_tracking_detection_gap: 0.015856666666666665
mean_frames_without_track_ids: 2.3785
mean_visible_joints_per_frame: 28.72195333333333
mean_keypoint_confidence: 0.62052831600877
mean_unique_track_ids_per_video: 6.1155
OCSORT
num_samples: 2000
empty_samples: 286
mean_detection_coverage: 0.7097066666666667
mean_tracking_coverage: 0.6956366666666667
mean_tracking_detection_gap: 0.014070000000000001
mean_frames_without_track_ids: 2.1105
mean_visible_joints_per_frame: 28.91218333333333
mean_keypoint_confidence: 0.6197117161977214
mean_unique_track_ids_per_video: 5.687
